# Pre + Post Compaction Demo Report

**Pre-compaction**: [OpenProvence](https://github.com/hotchpotch/open_provence) (sentence-level reranker-pruner)  
**Post-compaction**: [Headroom](https://pypi.org/project/headroom-ai/) (content-aware structural compression)  
**Local LLM**: Qwen2.5-7B-Instruct via Ollama  

This notebook loads the structured logs from the pipeline demo, visualises the results
with Plotly charts, and provides a narrative summary suitable for offline sharing.

## 1. Setup & Data Loading

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.io as pio

# render plots inline in the notebook
pio.renderers.default = "notebook"

from compaction.logger import load_logs
from evaluation.visualizations import (
    token_comparison_chart,
    latency_breakdown_chart,
    savings_chart,
    synergy_chart,
)

runs = load_logs()
print(f"Loaded {len(runs)} run(s) from logs/runs.json")

# Show latest 4 runs (one per mode)
latest_runs = runs[-4:] if len(runs) >= 4 else runs
print(f"Analysing {len(latest_runs)} modes: {[r['mode'] for r in latest_runs]}")

## 2. Summary Table

A quick overview of token counts, compression ratios, and latency for each compaction mode.

In [ ]:
summary_rows = []
for run in latest_runs:
    stages = run["stages"]
    inp = stages.get("input", {})
    pre = stages.get("pre_compaction", {})
    llm = stages.get("llm", {})
    post = stages.get("post_compaction", {})

    summary_rows.append({
        "Mode": run["mode"].replace("_", " ").title(),
        "Raw Input Tokens": inp.get("tokens_in", 0),
        "After Pre-Compaction": pre.get("tokens_out", "-") if pre else "-",
        "LLM Input Tokens": llm.get("tokens_in", 0),
        "LLM Output Tokens": llm.get("tokens_out", 0),
        "Pre Compression %": f"{pre.get('compression_ratio', 0):.1%}" if pre else "-",
        "Post Tokens Saved": post.get("extra", {}).get("tokens_saved", "-") if post else "-",
        "End-to-End Latency": f"{run.get('end_to_end_latency_ms', 0):,.0f}ms",
    })

df = pd.DataFrame(summary_rows)
df.style.set_caption("Compaction Pipeline Results").set_table_styles(
    [{"selector": "caption", "props": "font-size:1.2em; font-weight:bold;"}]
)

## 3. Token Comparison

How many tokens does each mode send to (and receive from) the LLM?

In [ ]:
fig = token_comparison_chart(latest_runs)
fig.show()

### Key Takeaway

OpenProvence's pre-compaction significantly reduces the number of tokens sent to the LLM
by pruning low-relevance sentences before inference. The LLM still produces comparable
output quality with fewer input tokens.

## 4. Latency Breakdown

Where does the time go? This chart breaks down end-to-end latency into
pre-compaction, LLM inference, and post-compaction stages.

In [ ]:
fig = latency_breakdown_chart(latest_runs)
fig.show()

### Key Takeaway

Pre-compaction adds ~150ms (after model warm-up) of reranking overhead, but reduces
LLM inference time because the model processes fewer tokens. The net effect can be
faster end-to-end than baseline for longer documents.

## 5. Savings vs Quality

Does compacting tokens hurt answer quality? This chart overlays token savings with
ROUGE-L quality scores.

In [ ]:
fig = savings_chart(latest_runs)
fig.show()

## 6. Synergy Analysis: Do Pre + Post Savings Compound?

A key question for production use: **when you combine pre-compaction with post-compaction,
are the savings additive, sub-additive, or synergistic?**

- **Additive** = sum of individual savings
- **Synergistic** = actual combined savings exceed the additive expectation
- **Sub-additive** = diminishing returns from combining both

In [ ]:
fig = synergy_chart(latest_runs)
fig.show()

### Synergy Interpretation

The synergy chart reveals how the two compaction stages interact:

- **If Pre + Post (Actual) > Pre + Post (Expected Additive)**: The techniques are
  synergistic — pre-compaction restructures context so post-compaction can compress
  even more effectively.
- **If roughly equal**: The savings are independent and additive.
- **If less**: There is overlap in what each technique removes.

## 7. Answers Comparison

How do the actual LLM answers differ across modes? This is the qualitative check to
accompany the quantitative metrics above.

In [ ]:
from IPython.display import Markdown, display

for run in latest_runs:
    mode = run["mode"].replace("_", " ").title()
    answer = run.get("answer_final", "(no answer)")
    display(Markdown(f"### {mode}\n\n{answer}\n\n---"))

## 8. Production Considerations

### OpenProvence (Pre-Compaction)
| Aspect | Detail |
|--------|--------|
| **How it works** | Cross-encoder reranker scores each sentence against the query, prunes low-scoring sentences |
| **Model size** | ~33M params (xsmall variant) — runs on CPU in ~150ms after warm-up |
| **Token savings** | 30–50% typical for document-length context |
| **Quality impact** | High — reranking preserves query-relevant information |
| **Production fit** | Excellent for RAG pipelines; language-agnostic; small resource footprint |

### Headroom (Post-Compaction)
| Aspect | Detail |
|--------|--------|
| **How it works** | Content-aware structural compression of message sequences |
| **Integration** | Acts on the full message chain (system + user + assistant) |
| **Token savings** | Variable — depends on message structure and content |
| **Reversibility** | Transforms are traceable and auditable |
| **Production fit** | Good for multi-turn conversations and long system prompts |

### Combined Strategy
For production pipelines processing documents through local LLMs:
1. **Pre-compaction** (OpenProvence) reduces context *before* inference → faster + cheaper
2. **Post-compaction** (Headroom) compresses the full conversation → smaller cache / history
3. The two stages target different aspects and their savings can be partially additive

## 9. Re-run the Pipeline (Optional)

To generate fresh data, run all four modes on a scenario and reload:

In [ ]:
# Uncomment the lines below to re-run the pipeline:

# from data.fetch_papers import load_scenarios
# from compaction.pipeline import run_all_modes
#
# scenarios = load_scenarios()
# s = scenarios[0]
#
# logger, new_runs = run_all_modes(
#     paper_id=s["id"],
#     paper_title=s["title"],
#     query=s["query"],
#     context=s["text"],
#     threshold=0.1,
# )
# logger.save()
# print("New run saved — re-run cells above to see updated charts.")